### np.polyfit

In [4]:
import math

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _transpose(mat):
    rows = len(mat)
    cols = len(mat[0]) if rows else 0
    return [[mat[i][j] for i in range(rows)] for j in range(cols)]


def _matmul(A, B):
    m, n, p = len(A), len(B[0]), len(B)
    return [[sum(A[i][k] * B[k][j] for k in range(p)) for j in range(n)] for i in range(m)]


def _gauss_solve(A, b):
    n = len(A)
    aug = [A[i][:] + [b[i]] for i in range(n)]

    for col in range(n):
        pivot = col
        for row in range(col + 1, n):
            if abs(aug[row][col]) > abs(aug[pivot][col]):
                pivot = row
        if abs(aug[pivot][col]) < 1e-12:
            raise ValueError("Singular or ill-conditioned system")
        aug[col], aug[pivot] = aug[pivot], aug[col]

        for row in range(col + 1, n):
            factor = aug[row][col] / aug[col][col]
            for j in range(col, n + 1):
                aug[row][j] -= factor * aug[col][j]

    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = aug[i][n]
        for j in range(i + 1, n):
            x[i] -= aug[i][j] * x[j]
        x[i] /= aug[i][i]
    return x


def np_polyfit(x, y, deg, rcond=None, full=False, w=None, cov=False):
    if deg < 0:
        raise ValueError("deg must be non-negative")

    sx = get_shape(x)
    sy = get_shape(y)

    if len(sx) != 1 or len(sy) != 1:
        raise ValueError("x and y must be 1-D sequences")
    if sx[0] != sy[0]:
        raise ValueError("x and y must have the same length")
    if len(x) == 0:
        raise ValueError("x and y cannot be empty")

    n = len(x)
    m = deg + 1

    if n < m:
        raise ValueError("the number of data points must be at least deg + 1")

    xs = [float(v) for v in x]
    ys = [float(v) for v in y]

    # Vandermonde matrix with descending powers
    A = [[xs[i] ** (deg - j) for j in range(m)] for i in range(n)]

    if w is not None:
        if get_shape(w) != (n,):
            raise ValueError("w must have the same length as x")
        ws = [float(v) for v in w]
        for i in range(n):
            scale = math.sqrt(ws[i])
            A[i] = [v * scale for v in A[i]]
            ys[i] *= scale

    At = _transpose(A)
    normal_A = _matmul(At, A)
    normal_b = [sum(At[i][k] * ys[k] for k in range(n)) for i in range(m)]

    coeffs = _gauss_solve(normal_A, normal_b)

    if full:
        fitted = [sum(coeffs[j] * (xs[i] ** (deg - j)) for j in range(m)) for i in range(n)]
        residuals = [ys[i] - fitted[i] for i in range(n)]
        ssr = sum(r * r for r in residuals)
        rank = m
        singular_values = []
        return coeffs, [ssr], rank, singular_values, rcond

    if cov:
        raise NotImplementedError("cov=True not supported in this notebook version")

    return coeffs 

### test cases 

In [5]:
print(np_polyfit([0, 1, 2], [1, 3, 7], 2))
print(np_polyfit([0, 1, 2, 3], [1, 2, 3, 4], 1))
print(np_polyfit([0, 1, 2, 3], [1, 4, 9, 16], 2))
print(np_polyfit([0, 1, 2, 3], [1, 4, 9, 16], 3))
print(np_polyfit([0, 1, 2, 3], [1, 4, 9, 16], 2, full=True)) 

[0.999999999999996, 1.0000000000000082, 0.9999999999999983]
[1.0, 1.0]
[1.0000000000000044, 1.9999999999999856, 1.0000000000000062]
[-3.285141540271194e-14, 1.0000000000001505, 1.9999999999998377, 1.0000000000000124]
([1.0000000000000044, 1.9999999999999856, 1.0000000000000062], [7.967495142732219e-29], 3, [], None)


In [6]:
print(np_polyfit([0, 1], [1, 2, 3], 1)) 

ValueError: x and y must have the same length

In [7]:
print(np_polyfit([0, 1], [1, 2], -1)) 

ValueError: deg must be non-negative

In [8]:
print(np_polyfit([], [], 1)) 

ValueError: x and y cannot be empty

In [9]:
print(np_polyfit([[0, 1]], [1, 2], 1)) 

ValueError: x and y must be 1-D sequences

In [10]:
print(np_polyfit([0, 1], [[1, 2]], 1)) 

ValueError: x and y must be 1-D sequences